# Neural-Agent Training on Kaggle

Train the proprietary NeuralAgent model on Kaggle's free GPU (T4, 15.6GB VRAM).

**Neural-Agent is PRIVATE and PAID — not published on PyPI.**

## Setup
1. Enable GPU: **Settings > Accelerator > GPU T4 x2**
2. Enable Internet: **Settings > Internet > On**
3. Run all cells

## Step 0: Upload Neural-Agent (one-time setup)

Neural-Agent is private. Upload it as a Kaggle dataset:
1. Go to kaggle.com → Datasets → New Dataset
2. Upload `neural-agent-kaggle.zip` (from NeuralDBG repo root)
3. Set dataset to **Private**
4. Copy dataset slug (e.g. `yourname/neural-agent-private`)
5. Update the install path in Step 1 if slug differs

## Step 1: Install Dependencies

In [ ]:
!pip install neuraldbg transformers peft trl bitsandbytes datasets accelerate -q
# Neural-Agent is PRIVATE — install from uploaded Kaggle dataset zip
# Upload neural-agent-kaggle.zip as a Kaggle dataset first (see Step 0 below)
!pip install /kaggle/input/neural-agent-private/neural-agent-kaggle.zip -q

## Step 2: Collect Training Triplets

NeuralDBG captures (events -> hypothesis -> fix) triplets from benchmark scenarios.

In [ ]:
!neuralagent collect --output /kaggle/working/triplets --max-per-category 100

## Step 3: Format Dataset for Training

In [ ]:
import json
from pathlib import Path

triplets_dir = Path("/kaggle/working/triplets")
formatted_dir = Path("/kaggle/working/formatted")
formatted_dir.mkdir(parents=True, exist_ok=True)

# Load all triplets
triplets = []
for f in triplets_dir.glob("*.jsonl"):
    with open(f) as fh:
        for line in fh:
            if line.strip():
                triplets.append(json.loads(line))

print(f"Loaded {len(triplets)} triplets")

# Convert to instruction format for SFT
def triplet_to_instruction(t):
    events = t.get("events", [])
    hypothesis = t.get("hypothesis", "")
    fix = t.get("fix", {})
    
    instruction = f"""Analyze these training events and provide a diagnosis:

Events: {json.dumps(events[:5], indent=2)}

What is the root cause and how to fix it?"""
    
    response = f"""Root cause: {hypothesis}

Fix: {json.dumps(fix, indent=2)}"""
    
    return {"instruction": instruction, "response": response}

# Split into train/val
import random
random.seed(42)
random.shuffle(triplets)
split = int(len(triplets) * 0.9)
train_triplets = triplets[:split]
val_triplets = triplets[split:]

# Save as JSONL
for name, data in [("train", train_triplets), ("val", val_triplets)]:
    path = formatted_dir / f"{name}.jsonl"
    with open(path, "w") as f:
        for t in data:
            f.write(json.dumps(triplet_to_instruction(t)) + "\n")
    print(f"Saved {len(data)} samples to {path}")

## Step 4: Train with QLoRA

Uses Qwen2-0.5B as base model with LoRA adapters. Fits in T4's 15.6GB VRAM.

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"PyTorch: {torch.__version__}")

In [ ]:
from neuralagent.train.lora import LoRAConfig, train

config = LoRAConfig(
    base_model="Qwen/Qwen2-0.5B",
    output_dir="/kaggle/working/checkpoints",
    dataset_path="/kaggle/working/formatted",
    num_epochs=3,
    batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    max_seq_length=1024,
    lora_r=8,
    lora_alpha=16,
    use_4bit=True,
    bf16=True,
    save_steps=50,
    logging_steps=10,
    eval_steps=50,
)

train(config)

## Step 5: Export to GGUF

In [ ]:
!neuralagent export /kaggle/working/checkpoints/final --output-dir /kaggle/working/export

## Step 6: Download Model

Download the trained model from Kaggle Output tab.

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/neuralagent-model", "zip", "/kaggle/working/export")
print("Model archived. Download from Kaggle Output tab.")